# Agentic RAG with LangGraph

In [ ]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
llm = ChatOpenAI(model="gpt-4.1", temperature=0)
embeddings = OpenAIEmbeddings()

In [7]:
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x17eb61a90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x17eb62510>, root_client=<openai.OpenAI object at 0x17e019010>, root_async_client=<openai.AsyncOpenAI object at 0x17eb62270>, model_name='gpt-4.1', temperature=0.0, model_kwargs={}, openai_api_key=Sec

## State Definition

In [9]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [12]:
### Sample Document and VectorStore
# Sample documents for demonstration

sample_texts = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain with the ability to coordinate multiple chains across multiple steps of computation in a cyclic manner.",
    "RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. It retrieves relevant documents and uses them to provide context for generating more accurate responses.",
    "Vector databases store high-dimensional vectors and enable efficient similarity search. They are commonly used in RAG systems to find relevant documents based on semantic similarity.",
    "Agentic systems are AI systems that can take actions, make decisions, and interact with their environment autonomously. They often use planning and reasoning capabilities.",
]

documents = [Document(page_content=text) for text in sample_texts]

# Create Vector Store

vector_store = FAISS.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(k=3)

### Agents function

In [13]:
def decide_retrieval(state: AgentState) -> AgentState:
    question = state["question"]

    # Simple heuristic: if question contains certain keywords then retrieve
    retrieval_keywords = ["what", "how", "explain", "tell me", "describe"]
    needs_retrieval = any(keyword in question.lower() for keyword in retrieval_keywords)

    return {**state, "needs_retrieval": needs_retrieval}

In [14]:
def retrieve_documents(state: AgentState) -> AgentState:

    question = state["question"]
    documents = retriever.invoke(question)

    return {**state, "documents": documents}

In [ ]:
def generate_answer(state: AgentState) -> AgentState:

    question = state["question"]
    documents = state.get("documents", [])

    if documents:
        # RAG Approach use documents as context
        context = "\n\n".join([doc.page_content for doc in documents])
        prompt = f""" Based on the following context, answer the question
        Context: {context}

        Question: {question}

        Answer:
        """
    else:
        # Direct response without retrieval
        prompt = f"Answer the following question {question}"

    response = llm.invoke(prompt)
    answer = response.content

    return {**state, "answer": answer}

### Conditonal Logic

### Build the Graph